In [ ]:
import os

for dirname, dirnames, filenames in os.walk('/kaggle/input/'):
    # Print the current folder
    print(f'Folder: {dirname}')
    
    # Print all files in this folder with an indentation
    for filename in filenames:
        print(f'    File: {filename}')

In [1]:
!pip install timm albumentations -q

In [2]:
import os, cv2, numpy as np, pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import random
from torch.utils.data import Sampler

In [3]:
DATA_DIR = "/kaggle/input/competitions/animal-part-recognition/animal-part"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

train_df.columns = ["image", "Id"]
sample_sub.columns = ["image", "Id"]

train_df = train_df[train_df["Id"] != "other"].reset_index(drop=True)

In [4]:
label2idx = {k: i for i, k in enumerate(train_df['Id'].unique())}
idx2label = {v: k for k, v in label2idx.items()}

train_df["label"] = train_df["Id"].map(label2idx)

NUM_CLASSES = len(label2idx)
print("Num classes:", NUM_CLASSES)

Num classes: 4570


In [5]:
class AnimalDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['image']}"
        
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        if self.is_test:
            return img, row['image']
        else:
            return img, row['label']

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 256

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Affine(
        translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)}, 
        scale=(0.9, 1.1), 
        rotate=(-15, 15), 
        p=0.5
    ),
    A.CoarseDropout(
        num_holes_range=(1, 8), 
        hole_height_range=(8, 32), 
        hole_width_range=(8, 32), 
        p=0.2
    ),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [9]:
class BalancedBatchSampler(Sampler):
    def __init__(self, labels, n_classes=16, n_samples=4):
        self.labels = np.array(labels)
        self.label_to_indices = {}

        for label in np.unique(self.labels):
            self.label_to_indices[label] = np.where(self.labels == label)[0]

        self.labels_set = list(self.label_to_indices.keys())
        self.n_classes = n_classes
        self.n_samples = n_samples

        self.batch_size = self.n_classes * self.n_samples
        self.num_batches = len(self.labels) // self.batch_size

    def __iter__(self):
        for _ in range(self.num_batches):
            classes = random.sample(self.labels_set, self.n_classes)
            batch = []

            for c in classes:
                indices = self.label_to_indices[c]
                batch.extend(np.random.choice(indices, self.n_samples, replace=True))

            yield batch

    def __len__(self):
        return self.num_batches

In [12]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_dataset = AnimalDataset(train_df, f"{DATA_DIR}/train", train_transforms)
val_dataset   = AnimalDataset(val_df,   f"{DATA_DIR}/train", val_transforms)

sampler = BalancedBatchSampler(train_df["label"].values, 16, 4)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=sampler,
    num_workers=2
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [13]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s = s
        self.m = m

    def forward(self, x, labels):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        cosine = cosine.clamp(-1, 1)
        phi = cosine - self.m

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1,1), 1)

        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s

In [14]:
class Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        self.embedding = nn.Linear(self.backbone.num_features, 512)
        self.bn = nn.BatchNorm1d(512)
        self.arc = ArcMarginProduct(512, num_classes, s=30.0, m=0.3)

    def forward(self, x, labels=None, use_arc=True):
        x = self.backbone(x)
        emb = self.embedding(x)
        emb = self.bn(emb)
        emb = F.normalize(emb)
        
        cosine = F.linear(emb, F.normalize(self.arc.weight))
        
        if labels is not None:
            if use_arc:
                logits = self.arc(emb, labels)
            else:
                logits = cosine * 30.0
            
            return emb, logits, cosine
        
        return emb

In [ ]:
def get_map5_score(indices, labels):
    scores = 0.0
    for i in range(len(labels)):
        where = torch.where(indices[i] == labels[i])[0]
        if len(where) > 0:
            rank = where[0].item()
            scores += 1.0 / (rank + 1)
    return scores

In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(NUM_CLASSES).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_map5 = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    use_arc = epoch >= 3
    
    pbar = tqdm(train_loader, total=len(train_loader), desc=f"Epoch {epoch} [Train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        emb, logits, cosine = model(imgs, labels, use_arc=use_arc)
        loss = criterion(logits, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = cosine.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
        pbar.set_description(f"Loss: {loss.item():.4f} | Acc: {train_correct/train_total:.4f}")
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total, val_map5_total = 0, 0, 0, 0.0

    pbar_val = tqdm(val_loader, total=len(val_loader), desc=f"Epoch {epoch} [Val]")
    with torch.no_grad():
        for imgs, labels in pbar_val:
            imgs, labels = imgs.to(device), labels.to(device)

            emb, logits, cosine = model(imgs, labels, use_arc=use_arc)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            
            _, top5_indices = torch.topk(cosine, k=5, dim=1)
            val_map5_total += get_map5_score(top5_indices, labels)
            
            val_correct += (top5_indices[:, 0] == labels).sum().item()
            val_total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total
    val_map5 = val_map5_total / val_total
    
    if val_map5 > best_val_map5:
        best_val_map5 = val_map5
        torch.save(model.state_dict(), "best_model.pth")
        tqdm.write(f"⭐ New Best Model Saved! MAP@5: {val_map5:.4f}")

    scheduler.step()
    
    tqdm.write(
        f"--- Epoch {epoch} Final ---\n"
        f"Train: Loss {avg_train_loss:.4f} | Acc {train_acc:.4f}\n"
        f"Val:   Loss {avg_val_loss:.4f} | Acc {val_acc:.4f} | MAP@5 {val_map5:.4f}\n"
        f"LR:    {optimizer.param_groups[0]['lr']:.6f}\n"
    )

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Loss: 8.2957 | Acc: 0.0267: 100%|██████████| 100/100 [01:16<00:00,  1.30it/s]
Epoch 0 [Val]: 100%|██████████| 26/26 [00:25<00:00,  1.01it/s]


⭐ New Best Model Saved! MAP@5: 0.0018
--- Epoch 0 Final ---
Train: Loss 8.7024 | Acc 0.0267
Val:   Loss 8.9519 | Acc 0.0012 | MAP@5 0.0018
LR:    0.000099



Loss: 7.2446 | Acc: 0.0892: 100%|██████████| 100/100 [01:18<00:00,  1.28it/s]
Epoch 1 [Val]: 100%|██████████| 26/26 [00:18<00:00,  1.42it/s]


⭐ New Best Model Saved! MAP@5: 0.0048
--- Epoch 1 Final ---
Train: Loss 7.7366 | Acc 0.0892
Val:   Loss 8.6834 | Acc 0.0019 | MAP@5 0.0048
LR:    0.000096



...

⭐ New Best Model Saved! MAP@5: 0.0289
--- Epoch 12 Final ---
Train: Loss 10.6228 | Acc 0.6797
Val:   Loss 16.9772 | Acc 0.0187 | MAP@5 0.0289
LR:    0.000004



Loss: 9.1860 | Acc: 0.6948: 100%|██████████| 100/100 [01:18<00:00,  1.27it/s]
Epoch 13 [Val]: 100%|██████████| 26/26 [00:17<00:00,  1.45it/s]


--- Epoch 13 Final ---
Train: Loss 10.4818 | Acc 0.6948
Val:   Loss 16.9714 | Acc 0.0149 | MAP@5 0.0280
LR:    0.000001



Loss: 12.0578 | Acc: 0.6711: 100%|██████████| 100/100 [01:18<00:00,  1.27it/s]
Epoch 14 [Val]: 100%|██████████| 26/26 [00:17<00:00,  1.45it/s]

--- Epoch 14 Final ---
Train: Loss 10.6803 | Acc 0.6711
Val:   Loss 16.9695 | Acc 0.0168 | MAP@5 0.0278
LR:    0.000000



In [29]:
train_dataset = AnimalDataset(train_df, f"{DATA_DIR}/train", val_transforms)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)

model.eval()
train_embeddings = []
train_labels = []

with torch.no_grad():
    for imgs, labels in tqdm(train_loader):
        imgs = imgs.to(device)
        emb = model(imgs)
        train_embeddings.append(emb.cpu())
        train_labels.extend(labels.numpy())

train_embeddings = torch.cat(train_embeddings)
train_labels = np.array(train_labels)

100%|██████████| 101/101 [01:11<00:00,  1.42it/s]


In [30]:
test_df = sample_sub.copy()

test_dataset = AnimalDataset(test_df, f"{DATA_DIR}/test", val_transforms, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [31]:
predictions = []

with torch.no_grad():
    for imgs, names in tqdm(test_loader):
        imgs = imgs.to(device)
        emb = model(imgs).cpu()
        
        sim = F.cosine_similarity(
            emb.unsqueeze(1),
            train_embeddings.unsqueeze(0),
            dim=2
        )
        
        values, indices = sim.topk(20, dim=1)
        
        for i in range(len(names)):
            seen = set()
            unique_ids = []
            
            for score, j in zip(values[i], indices[i]):
                label = idx2label[train_labels[j]]
                
                if label not in seen:
                    unique_ids.append(label)
                    seen.add(label)
                
                if len(unique_ids) == 5:
                    break
            
            if values[i][0] < 0.5:
                unique_ids = ["other"] + unique_ids[:4]
            
            while len(unique_ids) < 5:
                unique_ids.append("other")
            
            predictions.append(" ".join(unique_ids))

100%|██████████| 80/80 [02:57<00:00,  2.21s/it]


In [32]:
import os

def get_next_submission_name():
    i = 1
    while os.path.exists(f"submission_{i}.csv"):
        i += 1
    return f"submission_{i}.csv"

filename = get_next_submission_name()
print("Saving to:", filename)

Saving to: submission_1.csv


In [33]:
sample_sub["Id"] = predictions
sample_sub.to_csv(filename, index=False)

In [ ]:
# ml

In [ ]:
def get_clean_embeddings(model, dataloader, device, is_test=False):
    model.eval()
    embeddings = []
    meta_info = []
    
    with torch.no_grad():
        for imgs, data in tqdm(dataloader):
            imgs = imgs.to(device)
            x = model.backbone(imgs)
            emb = model.embedding(x)
            emb = model.bn(emb)
            emb = F.normalize(emb)
            
            embeddings.append(emb.cpu().numpy())
            meta_info.extend(data if is_test else data.numpy())
            
    return np.vstack(embeddings), meta_info

model.load_state_dict(torch.load("best_model.pth"))

train_dataset_clean = AnimalDataset(train_df, f"{DATA_DIR}/train", val_transforms)
train_loader_clean = DataLoader(train_dataset_clean, batch_size=64, shuffle=False)
X_train_knn, y_train_labels = get_clean_embeddings(model, train_loader_clean, device)

X_test_knn, test_image_names = get_clean_embeddings(model, test_loader, device, is_test=True)

100%|██████████| 80/80 [00:53<00:00,  1.49it/s]


In [ ]:
import torch.nn.functional as F

train_emb_ts = torch.from_numpy(X_train_knn).to(device)
test_emb_ts = torch.from_numpy(X_test_knn).to(device)

submission_data = []

for i in tqdm(range(len(test_emb_ts)), desc="Dự đoán"):
    sim = torch.mm(test_emb_ts[i:i+1], train_emb_ts.t()) # [1, N_train]
    
    values, indices = torch.topk(sim, k=min(50, len(train_emb_ts)), dim=1)
    
    values = values[0].cpu().numpy()
    indices = indices[0].cpu().numpy()
    
    unique_ids = []
    seen = set()
    
    for val, idx in zip(values, indices):
        label_id = idx2label[y_train_labels[idx]]
        if label_id not in seen:
            unique_ids.append(label_id)
            seen.add(label_id)
        if len(unique_ids) == 5:
            break

    if values[0] < 0.35: 
        if "other" not in unique_ids:
            unique_ids = ["other"] + unique_ids[:4]

    submission_data.append({
        "Image": test_image_names[i],
        "Id": " ".join(unique_ids)
    })

df_sub = pd.DataFrame(submission_data)
df_sub.to_csv("submission_final_v6.csv", index=False)
print("✅ Hoàn thành! Kiểm tra file: ", df_sub.head(2))

Dự đoán: 100%|██████████| 5072/5072 [00:01<00:00, 4608.52it/s]


✅ Hoàn thành! Kiểm tra file:             Image                                                 Id
0  cfb8c68dc.jpg      other w_2db0644 w_208db25 w_ef83760 w_700ebb4
1  66bf04895.jpg  w_e94b46a w_b27b6c6 w_1260eb5 w_3815890 w_ec32fa6


In [ ]:
raw_train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
raw_train_df.columns = ["image", "Id"]

full_train_df = raw_train_df[raw_train_df["Id"] != "other"].reset_index(drop=True)

full_train_df["label"] = full_train_df["Id"].map(label2idx)

if full_train_df["label"].isnull().any():
    print("⚠️ Cảnh báo: Có một số Id không khớp với từ điển label2idx!")
    full_train_df = full_train_df.dropna(subset=["label"])
    full_train_df["label"] = full_train_df["label"].astype(int)

full_train_dataset = AnimalDataset(full_train_df, f"{DATA_DIR}/train", val_transforms)
train_loader_full = DataLoader(full_train_dataset, batch_size=64, shuffle=False)

print("🚀 Bắt đầu trích xuất ngân hàng vector...")
X_train_knn, y_train_labels = get_clean_embeddings(model, train_loader_full, device)

X_test_knn, test_image_names = get_clean_embeddings(model, test_loader, device, is_test=True)

test_ts = torch.from_numpy(X_test_knn).to(device)
train_ts = torch.from_numpy(X_train_knn).to(device)

final_preds = []
for i in tqdm(range(len(test_ts))):
    sim = torch.mm(test_ts[i:i+1], train_ts.t()) # [1, N_train]
    
    vals, idxs = torch.topk(sim, k=min(100, len(train_ts)), dim=1)
    vals = vals[0].cpu().numpy()
    idxs = idxs[0].cpu().numpy()
    
    unique_ids = []
    seen = set()
    
    for v, idx in zip(vals, idxs):
        label_name = idx2label[y_train_labels[idx]]
        if label_name not in seen:
            unique_ids.append(label_name)
            seen.add(label_name)
        if len(unique_ids) == 5: break

    if "other" not in unique_ids:
        unique_ids = ["other"] + unique_ids[:4]
        
    final_preds.append(" ".join(unique_ids))

# Lưu file
df_sub = pd.DataFrame({"Image": test_image_names, "Id": final_preds})
df_sub.to_csv("submission_fix_knn.csv", index=False)

🚀 Bắt đầu trích xuất ngân hàng vector...


100%|██████████| 5072/5072 [00:01<00:00, 3398.76it/s]
